In [1]:
!pip install psycopg2-binary sqlalchemy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 8.3 MB/s eta 0:00:00m eta 0:00:010:00:01


In [3]:
!pip install python-dotenv

In [11]:
from dotenv import load_dotenv
import os 
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
import pandas as pd

load_dotenv("../.env")
DB_PASSWORD = os.getenv('DB_PASSWORD')
print('DB_loaded', DB_PASSWORD is not None)

DB_loaded True


In [25]:
connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username="shwetasehrawat",
    password=DB_PASSWORD,
    host="localhost",
    port=5432,
    database="shweta_housing")
engine= create_engine(connection_url)
with engine.connect() as conn:
    print("postgresql connection successful")

postgresql connection successful


In [27]:
city_metrics = pd.read_csv('../data/clean/city_metrics_cities.csv')
city_metrics_national = pd.read_csv('../data/clean/city_metrics_national.csv')
rents_sales = pd.read_csv('../data/clean/rents_sales_cities.csv')
rents_sales_national = pd.read_csv('../data/clean/rents_sales_national.csv')
tom = pd.read_csv('../data/clean/etw_efh_tom_cities.csv')
tom_national = pd.read_csv('../data/clean/etw_efh_tom_cities_national.csv')
bb_volume = pd.read_csv('../data/clean/bundesbank_lending_volume.csv')
bb_rate_all = pd.read_csv('../data/clean/bundesbank_interest_rate_all.csv')
bb_rate_10yr = pd.read_csv('../data/clean/bundesbank_interest_rate_10yr_fixed.csv')
tables = {"city_metrics":city_metrics,
          "city_metrics_national":city_metrics_national,
         "rents_sales":rents_sales,
         "rents_sales_national":rents_sales_national,
         "tom":tom,
         "tom_national":tom_national,
         "bb_volume":bb_volume,
         "bb_rate_all":bb_rate_all,
         "bb_rate_10yr":bb_rate_10yr,}
for tablename, df in tables.items():
    df.to_sql(tablename, engine, if_exists = "replace", index=False)
    print(f"loaded {tablename} : {df.shape}")

loaded city_metrics : (18204, 13)
loaded city_metrics_national : (492, 13)
loaded rents_sales : (2704, 7)
loaded rents_sales_national : (134, 7)
loaded tom : (2736, 8)
loaded tom_national : (114, 8)
loaded bb_volume : (282, 3)
loaded bb_rate_all : (282, 3)
loaded bb_rate_10yr : (282, 3)


In [29]:
result = pd.read_sql("select count(*) from city_metrics ", engine)
print(result)

   count
0  18204


In [35]:
q1 = """
select "City", round(avg("AVG_PRICE_SQM")::NUMERIC,2) as average_price, count(*) as num_quarters
from city_metrics
where "Granularity"='quarterly' and "Inflation_adjusted"=1
group by "City"
order by average_price desc;
"""
city_avg_price = pd.read_sql(q1,engine)
print(city_avg_price)

                 City  average_price  num_quarters
0             München          16.25            58
1   Frankfurt am Main          12.68            58
2           Stuttgart          11.86            58
3             Hamburg          11.12            58
4                Köln          10.74            58
5          Düsseldorf          10.11            58
6              Berlin          10.04            58
7           Wiesbaden           9.84            58
8             Münster           9.70            58
9           Karlsruhe           9.65            58
10               Bonn           9.61            58
11            Potsdam           9.55            58
12           Augsburg           9.51            58
13           Nürnberg           9.06            58
14           Mannheim           8.97            58
15             Aachen           8.34            58
16           Hannover           8.26            58
17   Rhein-Erft-Kreis           8.23            58
18             Bremen          

In [47]:
q2 = """ 
with city_avg as (
select "City", round(avg("AVG_PRICE_SQM")::NUMERIC,2) as average_price
from city_metrics
where "Granularity"='quarterly' and "Inflation_adjusted"=1
group by "City"
)
select "City", average_price, rank() over (order by average_price desc) as price_rank from city_avg 
order by price_rank;
"""
city_price_ranked = pd.read_sql(q2, engine)
print(city_price_ranked)

                 City  average_price  price_rank
0             München          16.25           1
1   Frankfurt am Main          12.68           2
2           Stuttgart          11.86           3
3             Hamburg          11.12           4
4                Köln          10.74           5
5          Düsseldorf          10.11           6
6              Berlin          10.04           7
7           Wiesbaden           9.84           8
8             Münster           9.70           9
9           Karlsruhe           9.65          10
10               Bonn           9.61          11
11            Potsdam           9.55          12
12           Augsburg           9.51          13
13           Nürnberg           9.06          14
14           Mannheim           8.97          15
15             Aachen           8.34          16
16           Hannover           8.26          17
17   Rhein-Erft-Kreis           8.23          18
18             Bremen           8.02          19
19             Lübec

In [53]:
q3 = """
with first_last as (
select "City", first_value("Rent_Index") over(Partition by "City" 
order by "Year", "Quarter") as first_rent,
last_value ("Rent_Index") over (partition by "City"
order by "Year", "Quarter" rows between unbounded preceding and unbounded following) as last_rent,
first_value("Sales_Index") over (Partition by "City"
order by "Year", "Quarter") as first_sales,
last_value ("Sales_Index") over (partition by "City"
order by "Year", "Quarter" rows between unbounded preceding and unbounded following) as last_sales  
from rents_sales
where "Granularity" = 'quarterly' and "Inflation_adjusted"=1),
growth as ( select distinct"City", round(((last_rent - first_rent)/first_rent*100)::numeric, 1) as rent_growth_pct,
round(((last_sales - first_sales)/first_sales*100)::numeric,1 ) as sales_growth_pct
from first_last)
select "City", rent_growth_pct, sales_growth_pct, round((sales_growth_pct - rent_growth_pct)::numeric,1) as gap, 
rank() over (order by (sales_growth_pct - rent_growth_pct) desc) as gap_rank 
from growth 
order by gap_rank;
"""
divergence = pd.read_sql(q3, engine)
print(divergence)

                 City  rent_growth_pct  sales_growth_pct    gap  gap_rank
0    Rhein-Erft-Kreis             19.0             124.3  105.3         1
1             Potsdam             29.2              99.0   69.8         2
2              Berlin             48.0             112.0   64.0         3
3                Bonn             14.8              74.8   60.0         4
4             Leipzig             33.6              92.8   59.2         5
5                Köln             22.1              81.1   59.0         6
6   Frankfurt am Main             15.2              68.0   52.8         7
7             Dresden             16.4              65.9   49.5         8
8      Kreis Mettmann             13.8              62.8   49.0         9
9             Münster             19.6              68.1   48.5        10
10           Dortmund             22.8              71.3   48.5        10
11          Stuttgart             19.1              64.3   45.2        12
12         Düsseldorf             18.8

In [57]:
q4 = """
with tom_yearly as (
select "Year", round(avg("TOM_4q")::numeric,1) as avg_tom_days from tom
group by "Year"),
rate_yearly as (select cast(left("Date",4) as integer) as year, round(avg("Interest_rate")::numeric,2) as avg_interest_rate 
from bb_rate_all 
group by cast(left("Date",4) as integer)
)
select t."Year", t.avg_tom_days, r.avg_interest_rate from tom_yearly as t
join rate_yearly as r on t."Year" = r.year 
order by t."Year";
"""
liquidity_vs_rate = pd.read_sql(q4, engine)
print(liquidity_vs_rate)


    Year  avg_tom_days  avg_interest_rate
0   2012          85.5               3.07
1   2013         100.0               2.76
2   2014          95.0               2.50
3   2015          87.8               1.95
4   2016          82.7               1.76
5   2017          80.2               1.83
6   2018          76.2               1.87
7   2019          77.6               1.52
8   2020          72.3               1.25
9   2021          65.7               1.26
10  2022          66.9               2.52
11  2023          87.7               4.00
12  2024         104.7               3.80
13  2025          92.5               3.67
14  2026          89.2               3.83


In [71]:
query_pressure = """
WITH price_growth AS (
    SELECT
        "City",
        FIRST_VALUE("AVG_PRICE_SQM") OVER (PARTITION BY "City" ORDER BY "Year", "Quarter") AS first_price,
        LAST_VALUE("AVG_PRICE_SQM") OVER (
            PARTITION BY "City" ORDER BY "Year", "Quarter"
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        ) AS last_price
    FROM city_metrics
    WHERE "Granularity" = 'quarterly' AND "Inflation_adjusted" = 1
),
price_growth_final AS (
    SELECT DISTINCT
        "City",
        ROUND(((last_price - first_price) / first_price * 100)::numeric, 1) AS price_growth_pct
    FROM price_growth
),
rent_sales_gap AS (
    SELECT
        "City",
        FIRST_VALUE("Rent_Index") OVER (PARTITION BY "City" ORDER BY "Year", "Quarter") AS first_rent,
        LAST_VALUE("Rent_Index") OVER (
            PARTITION BY "City" ORDER BY "Year", "Quarter"
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        ) AS last_rent,
        FIRST_VALUE("Sales_Index") OVER (PARTITION BY "City" ORDER BY "Year", "Quarter") AS first_sales,
        LAST_VALUE("Sales_Index") OVER (
            PARTITION BY "City" ORDER BY "Year", "Quarter"
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        ) AS last_sales
    FROM rents_sales
    WHERE "Granularity" = 'quarterly' AND "Inflation_adjusted" = 1
),
gap_final AS (
    SELECT DISTINCT
        "City",
        ROUND((((last_sales - first_sales) / first_sales * 100) - ((last_rent - first_rent) / first_rent * 100))::numeric, 1) AS rent_price_gap
    FROM rent_sales_gap
)
SELECT
    p."City",
    p.price_growth_pct,
    g.rent_price_gap,
    RANK() OVER (ORDER BY (p.price_growth_pct + g.rent_price_gap) DESC) AS pressure_rank
FROM price_growth_final p
JOIN gap_final g ON p."City" = g."City"
ORDER BY pressure_rank;
"""

pressure_sql = pd.read_sql(query_pressure, engine)
print(pressure_sql)

                 City  price_growth_pct  rent_price_gap  pressure_rank
0    Rhein-Erft-Kreis              25.3           105.2              1
1              Berlin              55.5            64.0              2
2             Potsdam              39.3            69.8              3
3             Leipzig              46.6            59.2              4
4                Köln              30.7            59.0              5
5                Bonn              20.0            60.1              6
6             München              36.1            41.0              7
7             Münster              28.2            48.5              8
8            Dortmund              27.6            48.5              9
9              Lübeck              40.7            35.3             10
10  Frankfurt am Main              19.7            52.9             11
11            Dresden              21.4            49.5             12
12          Stuttgart              25.1            45.3             13
13    